In [13]:
import os
import csv
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

In [4]:
csv_path = "train.csv"
rows = []
with open(csv_path, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        rows.append(row)

print(f"Wczytano {len(rows)} przykładów")

Wczytano 12672 przykładów


In [6]:
import cv2
import numpy as np

def lbp_hist(img_path, size=(64, 64)):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    if img is None:
        raise ValueError("Nie można wczytać obrazu")

    img = cv2.resize(img, size)

    h, w = img.shape
    lbp = np.zeros((h, w), dtype=np.uint8)

    neighbors = [
        (-1, -1), (-1, 0), (-1, 1),
        (0, 1),
        (1, 1), (1, 0), (1, -1),
        (0, -1)
    ]

    for y in range(1, h - 1):
        for x in range(1, w - 1):
            center = img[y, x]
            code = 0

            for i, (dy, dx) in enumerate(neighbors):
                if img[y + dy, x + dx] >= center:
                    code |= (1 << i)

            lbp[y, x] = code

    hist = np.zeros(256, dtype=np.float32)

    for value in lbp.ravel():
        hist[value] += 1

    hist /= (hist.sum() + 1e-7)

    return hist

In [7]:
X = []
y = []
valid_paths = []
skipped = 0

for row in rows:
    img_path = row["image_path"]
    label = int(row["label"])
    
    if not os.path.exists(img_path):
        print(f"Pominięty nieistniejący: {img_path}")
        skipped += 1
        continue
    
    hist = lbp_hist(img_path)
    X.append(hist)
    y.append(label)
    valid_paths.append(img_path)

X = np.array(X)
y = np.array(y)

print(f"W przygotowaniu: {len(X)} przykładów")
print(f"Pominiono: {skipped}")
print(f"Wymiary X: {X.shape}")

W przygotowaniu: 12672 przykładów
Pominiono: 0
Wymiary X: (12672, 256)


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

Train: 8870, Test: 3802


In [11]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

train_acc = rf.score(X_train, y_train)
test_acc = rf.score(X_test, y_test)

print("\n" + "="*50)
print("WYNIKI AKURACJI")
print("="*50)
print(f"Akuracja train: {train_acc:.3f}")
print(f"Akuracja test:  {test_acc:.3f}")


WYNIKI AKURACJI
Akuracja train: 1.000
Akuracja test:  0.691


In [17]:
y_pred = rf.predict(X_test)
print("\n" + "="*50)
print("CLASSIFICATION REPORT (test)")
print("="*50)
print(classification_report(y_test, y_pred, zero_division=0))


CLASSIFICATION REPORT (test)
              precision    recall  f1-score   support

           0       0.69      1.00      0.82      2616
           1       0.00      0.00      0.00        59
           2       0.00      0.00      0.00        17
           3       0.00      0.00      0.00        88
           4       0.00      0.00      0.00        57
           5       0.00      0.00      0.00        58
           6       0.50      0.01      0.01       347
           7       1.00      0.02      0.03        59
           8       0.00      0.00      0.00        17
           9       1.00      0.01      0.02        89
          10       0.00      0.00      0.00        60
          11       0.00      0.00      0.00        60
          12       0.37      0.03      0.05       275

    accuracy                           0.69      3802
   macro avg       0.27      0.08      0.07      3802
weighted avg       0.59      0.69      0.57      3802



In [18]:
import joblib
joblib.dump(rf, "chess_rf_model.joblib")
print("Zapisano chess_rf_model.joblib")

Zapisano chess_rf_model.joblib


# Drugie podejście detekcja klasy 0 następnie detekcja typów figór

niestety dodanie 2 kroku klasyfikacji nie wpłyneło pozytywnie na klasyfikację pomimo poprawy balansu klas

In [19]:
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.validation import check_is_fitted
from sklearn.utils.multiclass import unique_labels


class DoubleForestClassifier(BaseEstimator, ClassifierMixin):
    """
    Hierarchiczny klasyfikator z 2 Random Forestami:
    
    Krok 1: empty (0) vs non-empty (1-12)
    Krok 2: jeśli non-empty → 12 klas figur (0-11, po przeliczeniu z 1-12)
    
   Parameters
    ----------
    n_estimators_1 : int, default=200
        Liczba drzew w modelu kroku 1 (empty vs non-empty)
    
    n_estimators_2 : int, default=200
        Liczba drzew w modelu kroku 2 (12 klas figur)
    
    max_depth_1 : int or None, default=None
        Maksymalna głębokość drzew w modelu kroku 1
    
    max_depth_2 : int or None, default=None
        Maksymalna głębokość drzew w modelu kroku 2
    
    random_state : int or None, default=42
        Seed dla losowości
    
    n_jobs : int or None, default=-1
        Liczba procesów (−1 = wszystkie)
    
    Attributes
    ----------
    rf_step1 : RandomForestClassifier
        Klasyfikator kroku 1 (empty vs non-empty)
    
    rf_step2 : RandomForestClassifier
        Klasyfikator kroku 2 (12 klas figur)
    
    """
    
    def __init__(
        self,
        n_estimators_1=200,
        n_estimators_2=200,
        max_depth_1=None,
        max_depth_2=None,
        random_state=42,
        n_jobs=-1
    ):
        self.n_estimators_1 = n_estimators_1
        self.n_estimators_2 = n_estimators_2
        self.max_depth_1 = max_depth_1
        self.max_depth_2 = max_depth_2
        self.random_state = random_state
        self.n_jobs = n_jobs
        
        self.rf_step1 = None
        self.rf_step2 = None
    
    def fit(self, X, y, **fit_params):
        X = np.array(X)
        y = np.array(y)
        
        y_step1 = np.where(y == 0, 0, 1)  # 0 → 0, 1-12 → 1
        self.rf_step1 = RandomForestClassifier(
            n_estimators=self.n_estimators_1,
            max_depth=self.max_depth_1,
            random_state=self.random_state,
            n_jobs=self.n_jobs
        )
        self.rf_step1.fit(X, y_step1)
        X_nonempty = X[y != 0]
        y_nonempty_orig = y[y != 0]  # 1-12
        y_step2 = y_nonempty_orig - 1  # 0-11
        
        self.rf_step2 = RandomForestClassifier(
            n_estimators=self.n_estimators_2,
            max_depth=self.max_depth_2,
            random_state=self.random_state,
            n_jobs=self.n_jobs
        )
        self.rf_step2.fit(X_nonempty, y_step2)
        
        self.n_classes_ = 13
        self.classes_ = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12])
        
        return self
    
    def predict(self, X):
        check_is_fitted(self, ['rf_step1', 'rf_step2'])
        
        X = np.array(X)
        
        pred_step1 = self.rf_step1.predict(X)
        y_pred = np.full(len(X), -1, dtype=int)
        
        empty_idx = np.where(pred_step1 == 0)[0]
        y_pred[empty_idx] = 0
        nonempty_idx = np.where(pred_step1 == 1)[0]
        if len(nonempty_idx) > 0:
            pred_step2 = self.rf_step2.predict(X[nonempty_idx])
            y_pred[nonempty_idx] = pred_step2 + 1
        
        return y_pred
    
    def predict_proba(self, X):
        check_is_fitted(self, ['rf_step1', 'rf_step2'])
        
        X = np.array(X)
        proba_step1 = self.rf_step1.predict_proba(X)  # (N, 2)
        
        proba = np.zeros((len(X), 13))
        
        proba[:, 0] = proba_step1[:, 0]
        
        nonempty_idx = np.where(proba_step1[:, 1] > 0)[0]
        if len(nonempty_idx) > 0:
            proba_step2 = self.rf_step2.predict_proba(X[nonempty_idx])
            proba[nonempty_idx, 1:] = proba_step1[nonempty_idx, 1] * proba_step2
        
        return proba
    
    def score(self, X, y):
        y_pred = self.predict(X)
        return np.mean(y_pred == y)
    
    def get_params(self, deep=True):
        return {
            'n_estimators_1': self.n_estimators_1,
            'n_estimators_2': self.n_estimators_2,
            'max_depth_1': self.max_depth_1,
            'max_depth_2': self.max_depth_2,
            'random_state': self.random_state,
            'n_jobs': self.n_jobs
        }
    def set_params(self, **params):
        for key, value in params.items():
            setattr(self, key, value)
        return self

In [21]:
doubleForest = DoubleForestClassifier()
doubleForest.fit(X_train, y_train)

train_acc = doubleForest.score(X_train, y_train)
test_acc = doubleForest.score(X_test, y_test)

print("\n" + "="*50)
print("WYNIKI AKURACJI")
print("="*50)
print(f"Akuracja train: {train_acc:.3f}")
print(f"Akuracja test:  {test_acc:.3f}")


WYNIKI AKURACJI
Akuracja train: 1.000
Akuracja test:  0.678


In [22]:
y_pred = doubleForest.predict(X_test)
print("\n" + "="*50)
print("CLASSIFICATION REPORT (test)")
print("="*50)
print(classification_report(y_test, y_pred, zero_division=0))


CLASSIFICATION REPORT (test)
              precision    recall  f1-score   support

           0       0.74      0.93      0.83      2616
           1       0.00      0.00      0.00        59
           2       0.00      0.00      0.00        17
           3       0.00      0.00      0.00        88
           4       0.00      0.00      0.00        57
           5       0.00      0.00      0.00        58
           6       0.23      0.20      0.21       347
           7       1.00      0.05      0.10        59
           8       0.00      0.00      0.00        17
           9       0.71      0.06      0.10        89
          10       0.00      0.00      0.00        60
          11       1.00      0.02      0.03        60
          12       0.27      0.20      0.23       275

    accuracy                           0.68      3802
   macro avg       0.30      0.11      0.12      3802
weighted avg       0.60      0.68      0.61      3802

